# Q-Cog quickstart

Q-Cog models human judgement, decision making and groups with **quantum probability**, runs everything on
**Qiskit**, and always shows what a **classical** account of the same situation predicts.

This notebook is deliberately simple: run the cells from top to bottom. Everything happens offline on a free
local simulator, and no installation is needed beyond the library's dependencies.

In [ ]:
# Make the library importable when the notebook runs from a checkout
# (do nothing if you installed the package with pip).
import pathlib, sys

for candidate in (pathlib.Path("../src"), pathlib.Path("src")):
    if (candidate / "qcog").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break

import qcog
print("Q-Cog", qcog.__version__)
print("models:", len(qcog.list_models()))
for slug in qcog.list_models():
    print("  ", slug)

## 1. Run one model

Every model is run with `.run(...)`. The output tells you what it predicts, why, and what it does not claim.

In [ ]:
model = qcog.get_model("question-order")
result = model.run()

print(result.summary)
print()
for finding in result.findings:
    print("-", finding)

## 2. Change a parameter

Parameters are keyword arguments; `model.params` lists them with their bounds. Here we make the two questions
less incompatible and the order effect shrinks.

In [ ]:
for angle in (0.2, 0.955, 1.4):
    run = model.run(angle_a=angle, sweeps=False)
    print(f"angle_a = {angle:5.3f}  ->  {run.summary}")

## 3. Compare with classical cognition

Every run carries a quantity-by-quantity comparison with a classical account of the same situation. Where the
classical value is exactly zero, a classical model is *forced* to say nothing there - so the model's number is
the whole content of that comparison.

In [ ]:
print("baseline:", result.classical["name"])
print()
for row in result.classical_comparison:
    print(f"{row['metric']:<46s} quantum {row['model']:+.3f}   classical {row['baseline']:+.3f}")
print()
print(result.classical["verdict"])

In [ ]:
# the comparison chart, with its accessible description underneath
comparison = model.figures(result)["model versus classical baseline"]
display(comparison)
print(comparison._qcog_description)

## 4. Look at the quantum circuit

The library compiles the model's own design into a Qiskit circuit - both orders of the two questions here - and
each drawing carries a plain-language description. The image itself is adjustable: style, scale and folding.

In [ ]:
designs = model.circuit_designs(result)
print("circuits:", list(designs))

import qcog.viz

circuits = model.circuit_images(result, style="iqp", scale=0.8)
for name, figure in circuits.items():
    display(figure)
print(circuits["A then B"]._qcog_description)

## 5. Run it on a simulator

The same model definition goes through the compiled circuit on Aer, with shot noise. The QQ equality - a
prediction with no free parameters - is then reported with its standard error.

In [ ]:
on_device = model.run(backend="aer", shots=8000, seed=7)
print(on_device.summary)
print()
print("backend:", on_device.backend, "| shots:", on_device.shots)
print(on_device.findings[1])

## 6. Save a report you can publish

One file, no JavaScript, every chart with a text description and the data table behind it.

In [ ]:
from qcog import write_model_report

path = write_model_report(model, result, "../out/notebook_report")
print("report written to", path)

## Where to go next

- `showcase/run_showcase.py` - a scripted tour with its captured output, figures and reports.
- `docs/project_page.md`, `docs/methodology.md`, `docs/accessibility.md` - what the models mean and how they are
  checked.
- `qcog list`, `qcog run <slug> --param name=value`, `qcog circuit <slug>`, `qcog report` - the same things from
  the command line.
- `streamlit run app/streamlit_app.py` - the interactive explorer, with dark, large-type, high-contrast and
  text-only modes.

One habit worth keeping: read each model's **caveats**. These models describe *response probabilities*; they do
not claim that the brain is a quantum computer, and the library says so wherever the risk of over-reading is
highest.